In [17]:

# Task 1: Data Gathering & Combination

# Import necessary libraries for data processing and HTML parsingimport sqlite3
import sqlite3
import pandas as pd
from bs4 import BeautifulSoup

In [ ]:
# ==========================================
# Task 1 - Step 1: Database Connection & Table Inspection
# ==========================================

# Establish a connection to the SQLite database file
conn = sqlite3.connect("Library Database . db")

# Retrieve the list of all table names existing in the database schema
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

# Display the retrieved table names
tables

,name
0,members
1,books
2,checkouts


In [18]:
# ==========================================
# Task 1 - Step 2: Fetch and Inspect Individual SQL Tables
# ==========================================

# Query and load all records from the 'members' table into a DataFrame
members = pd.read_sql_query("SELECT * FROM members", conn)

# Query and load all records from the 'checkouts' table into a DataFrame
checkouts = pd.read_sql_query("SELECT * FROM checkouts", conn)

# Query and load all records from the 'books' table into a DataFrame
books = pd.read_sql_query("SELECT * FROM books", conn)

# Display the first 5 rows of each table to inspect their structure and data types
print("MEMBERS:")
display(members.head())

print("\nCHECKOUTS:")
display(checkouts.head())

print("\nBOOKS:")
display(books.head())

MEMBERS:


,member_id,first_name,last_name,grade,neighborhood,membership_status,join_date
0,1001,Salma,Ibrahim,8.0,Maadi,Active,2023-04-05
1,1002,Fares,Saleh,9.0,Maadi,Active,None
2,1003,Bassel,Hegazy,6.0,Maadi,Active,2025-04-23
3,1004,Fares,Wahba,7.0,Maadi,inactive,2024-10-09
4,1005,Youssef,Halim,9.0,Maadi,Active,2024-05-05



CHECKOUTS:


,checkout_id,member_id,book_id,checkout_date,return_date
0,9263,1047,517,2024-10-21,2024-11-07
1,9340,1072,513,2025-08-24,2025-09-01
2,9231,1053,523,2024-02-04,2024-02-16
3,9129,1032,513,2025-06-21,2025-06-29
4,9370,1079,511,2025-11-11,2025-12-03



BOOKS:


,book_id,title,author
0,501,The Silver Kite,Amina Darwish
1,502,Desert Compass,Amina Darwish
2,503,The Lantern Maker,Adel Roushdy
3,504,Rooftop Astronomers,Adel Roushdy
4,505,Letters to the Nile,Aya Hafez


In [19]:
# ==========================================
# Task 1 - Step 3: Join Members and Checkouts Tables via SQL
# ==========================================

# Define SQL query to perform an INNER JOIN between members and checkouts tables
# This combines student demographic info with their corresponding checkout records using 'member_id'
query = """
SELECT
    m.member_id,
    m.first_name,
    m.last_name,
    m.grade,
    m.neighborhood,
    m.membership_status,
    c.checkout_id,
    c.book_id,
    c.checkout_date,
    c.return_date
FROM members AS m
INNER JOIN checkouts AS c
    ON m.member_id = c.member_id;
"""

# Execute the join query and store the result in a DataFrame
sql_data = pd.read_sql_query(query, conn)

# Inspect the first 5 rows of the merged SQL dataset
display(sql_data.head())

,member_id,first_name,last_name,grade,neighborhood,membership_status,checkout_id,book_id,checkout_date,return_date
0,1047,Sara,Rashad,NaN,Heliopolis,Inactive,9263,517,2024-10-21,2024-11-07
1,1072,Seif,Zaki,9.0,Zamalek,Active,9340,513,2025-08-24,2025-09-01
2,1053,Adam,Shafik,9.0,Heliopolis,Active,9231,523,2024-02-04,2024-02-16
3,1032,Nada,Zaki,7.0,Nasr City,Active,9129,513,2025-06-21,2025-06-29
4,1079,Rana,Osman,8.0,Shubra,Active,9370,511,2025-11-11,2025-12-03


In [ ]:
# ==========================================
# Task 1 - Step 4: Inspect SQL Dataset Dimensions & Missing Values
# ==========================================

# Print total number of rows/records in the merged SQL dataset
print("Rows:", len(sql_data))

# Print the list of all column names
print("Columns:", sql_data.columns.tolist())

# Calculate and display the total count of missing (NaN) values per column
print("Missing values:")
display(sql_data.isnull().sum())

Rows: 391
Columns: ['member_id', 'first_name', 'last_name', 'grade', 'neighborhood', 'membership_status', 'checkout_id', 'book_id', 'checkout_date', 'return_date']
Missing values:


member_id             0
first_name            0
last_name             0
grade                36
neighborhood          0
membership_status     0
checkout_id           0
book_id               0
checkout_date         0
return_date          65
dtype: int64

In [20]:
# ==========================================
# Task 1 - Step 5: Read and Inspect Book Catalog from JSON
# ==========================================

# Load book catalog metadata from JSON file into a DataFrame
catalog = pd.read_json("Book Catalog . Jason")

# Preview the first 5 records of the JSON catalog
display(catalog.head())

# Inspect column names in the JSON dataset
print("Catalog Columns:", catalog.columns.tolist())

,book_id,genre,pages,publication_year,publisher
0,501,Adventure,128,2017.0,Nile Press
1,502,Adventure,109,2018.0,Delta House
2,503,Historical,259,NaN,Nile Press
3,504,Science,319,2009.0,Cairo Young Readers
4,505,Historical,216,2024.0,Oasis Books


Catalog Columns: ['book_id', 'genre', 'pages', 'publication_year', 'publisher']


In [22]:
# ==========================================
# Task 1 - Step 6: Merge SQL Data with JSON Book Catalog
# ==========================================

# Perform a left join to combine the SQL checkouts dataset with the JSON catalog
# Matches records using 'book_id' to retain all checkout records while adding catalog metadata
combined = sql_data.merge(
    catalog,
    on="book_id",
    how="left"
)

# Inspect the first 5 rows of the combined dataset
display(combined.head())

,member_id,first_name,last_name,grade,neighborhood,membership_status,checkout_id,book_id,checkout_date,return_date,genre,pages,publication_year,publisher
0,1047,Sara,Rashad,NaN,Heliopolis,Inactive,9263,517,2024-10-21,2024-11-07,Mystery,338,2015.0,Delta House
1,1072,Seif,Zaki,9.0,Zamalek,Active,9340,513,2025-08-24,2025-09-01,Science,294,2021.0,Oasis Books
2,1053,Adam,Shafik,9.0,Heliopolis,Active,9231,523,2024-02-04,2024-02-16,Historical,276,2018.0,Oasis Books
3,1032,Nada,Zaki,7.0,Nasr City,Active,9129,513,2025-06-21,2025-06-29,Science,294,2021.0,Oasis Books
4,1079,Rana,Osman,8.0,Shubra,Active,9370,511,2025-11-11,2025-12-03,Historical,117,2016.0,Nile Press


In [21]:
# ==========================================
# Task 1 - Step 6: Merge SQL Data with JSON Catalog & Books Metadata
# ==========================================

# Perform a left join to combine SQL checkout records with the JSON book catalog using 'book_id'
combined = sql_data.merge(catalog, on="book_id", how="left")

# Merge the result with the SQL 'books' table to append book titles and authors
final_db_data = combined.merge(books, on="book_id", how="left")

# Inspect the first 5 rows of the fully merged database dataset
display(final_db_data.head())

,member_id,first_name,last_name,grade,neighborhood,membership_status,checkout_id,book_id,checkout_date,return_date,genre,pages,publication_year,publisher,title,author
0,1047,Sara,Rashad,NaN,Heliopolis,Inactive,9263,517,2024-10-21,2024-11-07,Mystery,338,2015.0,Delta House,Shadows on the Corniche,Hani Nagati
1,1072,Seif,Zaki,9.0,Zamalek,Active,9340,513,2025-08-24,2025-09-01,Science,294,2021.0,Oasis Books,Circuits for Beginners,Galal Mounir
2,1053,Adam,Shafik,9.0,Heliopolis,Active,9231,523,2024-02-04,2024-02-16,Historical,276,2018.0,Oasis Books,Footsteps in the Dust,Laila Shokry
3,1032,Nada,Zaki,7.0,Nasr City,Active,9129,513,2025-06-21,2025-06-29,Science,294,2021.0,Oasis Books,Circuits for Beginners,Galal Mounir
4,1079,Rana,Osman,8.0,Shubra,Active,9370,511,2025-11-11,2025-12-03,Historical,117,2016.0,Nile Press,Winter in Alexandria,Farida Anwar


In [23]:
# ==========================================
# Task 1 - Step 7: Parse HTML File & Combine All Data Sources
# ==========================================

from bs4 import BeautifulSoup

# Open and parse the Reading Kickoff HTML file
with open("Reading Kickoff Signups . html", "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f.read(), "html.parser")

# Extract the HTML table containing event signups
html_table = soup.find("table")

# Read the HTML table into a Pandas DataFrame
html_events = pd.read_html(str(html_table))[0]

# Concatenate all data sources (SQL DB + JSON Metadata + HTML Signups) into a single master DataFrame
full_dataset = pd.concat([final_db_data, html_events], ignore_index=True)

# Inspect the final combined dataset dimensions and first few rows
print("Total records gathered:", len(full_dataset))
display(full_dataset.head())

Total records gathered: 417


C:\Users\mohan\AppData\Local\Temp\ipykernel_13276\704057129.py:15: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  html_events = pd.read_html(str(html_table))[0]


,member_id,first_name,last_name,grade,neighborhood,membership_status,checkout_id,book_id,checkout_date,return_date,genre,pages,publication_year,publisher,title,author,Member ID,Book ID,Checkout Date
0,1047.0,Sara,Rashad,NaN,Heliopolis,Inactive,9263.0,517.0,2024-10-21,2024-11-07,Mystery,338.0,2015.0,Delta House,Shadows on the Corniche,Hani Nagati,NaN,NaN,NaN
1,1072.0,Seif,Zaki,9.0,Zamalek,Active,9340.0,513.0,2025-08-24,2025-09-01,Science,294.0,2021.0,Oasis Books,Circuits for Beginners,Galal Mounir,NaN,NaN,NaN
2,1053.0,Adam,Shafik,9.0,Heliopolis,Active,9231.0,523.0,2024-02-04,2024-02-16,Historical,276.0,2018.0,Oasis Books,Footsteps in the Dust,Laila Shokry,NaN,NaN,NaN
3,1032.0,Nada,Zaki,7.0,Nasr City,Active,9129.0,513.0,2025-06-21,2025-06-29,Science,294.0,2021.0,Oasis Books,Circuits for Beginners,Galal Mounir,NaN,NaN,NaN
4,1079.0,Rana,Osman,8.0,Shubra,Active,9370.0,511.0,2025-11-11,2025-12-03,Historical,117.0,2016.0,Nile Press,Winter in Alexandria,Farida Anwar,NaN,NaN,NaN


In [26]:
# Task 1 Reflection & Summary:
# - SQL JOIN: Combined the members and checkouts tables using member_id.
# - Pandas Merge: Merged checkout records with the JSON book catalog using book_id.
# - HTML Extraction: Extracted signup table from HTML page using BeautifulSoup and pd.read_html().

# Save the final combined dataset to CSV
full_dataset.to_csv("task1_combined_data.csv", index=False)
print("Saved task1_combined_data.csv successfully!")

Saved task1_combined_data.csv successfully!
